# CAELLUM — Colab dataset + training notebook

**ENV: Colab / Kaggle GPU (CUDA).** This notebook runs the *Colab half* of the CAELLUM
image-enhancement pipeline top-to-bottom:

1. install training deps,
2. get the self-contained scripts onto the box,
3. `gen_teacher.py` — make polished target PNGs across the categories,
4. `roughen.py` — turn each into a crude "kid's sketch" + write the paired dataset,
5. `train_caellum_ip2p.py` — InstructPix2Pix fine-tune (primary),
6. zip + download the trained weights.

Then continue on the **Neuron box** at step 6 of
[`RUNBOOK-image.md`](../RUNBOOK-image.md) to fuse the LoRA at compile time.

> **Two-environment note.** Colab is CUDA and does **not** have this repo. The `data/` and
> `train/` scripts are *self-contained* (they inline `CATEGORIES`/`SHAPE`/`STYLE` from
> `services/caellum/config.py`). The Neuron box is a *separate* env (no CUDA) and runs
> `compile.py` + `serve.py` instead — that part is not in this notebook.

**Before running:** Runtime → Change runtime type → **GPU** (T4 or better).

## Cell 1 — Install training deps

Pinned deps live in `train/requirements-train.txt`. If you've already uploaded the repo (next
cell), the file install works; otherwise the inline `pip install` below is the fallback. Either
way you get `diffusers` + `accelerate` + `transformers` + CUDA `torch` for gen + roughen + train.

In [ ]:
import os

# Preferred: install from the pinned requirements file (present after the upload/clone cell).
if os.path.exists('train/requirements-train.txt'):
    print('Installing from train/requirements-train.txt ...')
    !pip install -q -r train/requirements-train.txt
else:
    # Fallback inline install (keep roughly in sync with train/requirements-train.txt).
    print('requirements-train.txt not found yet; installing inline fallback ...')
    !pip install -q "diffusers>=0.27" "transformers>=4.38" "accelerate>=0.27" \
        "datasets>=2.18" "safetensors>=0.4" opencv-python-headless pillow numpy

import torch
print('torch', torch.__version__, '| cuda available:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'No GPU — set Runtime -> Change runtime type -> GPU.'

## Cell 2 — Get the scripts onto the box

Colab does **not** have this repo. Get the four self-contained Colab-side files here:
`data/gen_teacher.py`, `data/roughen.py`, `train/train_caellum_ip2p.py`, and
`train/requirements-train.txt`.

**Option A (clone the repo):** uncomment and set the URL.

**Option B (upload):** comment the clone out and use the file-picker upload instead, or drag the
files into Colab's file pane preserving the `data/` and `train/` folders.

The scripts are self-contained, so this is just "get the `.py` files here" — nothing imports the
repo's `config.py`.

In [ ]:
import os

# --- Option A: clone the repo (PUBLIC — no token needed) --------------------
REPO_URL = 'https://github.com/Jeremyliu-621/magicboard.git'
BRANCH = 'feat/caellum-ai-pipeline'   # the data/ + train/ scripts live on THIS branch, not main
if REPO_URL and not os.path.exists('data/gen_teacher.py'):
    !git clone --depth 1 -b {BRANCH} {REPO_URL} _repo
    !cp -r _repo/data _repo/train . 2>/dev/null || true

# --- Option B: upload the files by hand ------------------------------------
# from google.colab import files
# print('Upload data/gen_teacher.py, data/roughen.py, train/train_caellum_ip2p.py, train/requirements-train.txt')
# up = files.upload()   # then move them into data/ and train/ if needed

os.makedirs('data', exist_ok=True)
os.makedirs('train', exist_ok=True)
for f in ['data/gen_teacher.py', 'data/roughen.py', 'train/train_caellum_ip2p.py']:
    print(('OK  ' if os.path.exists(f) else 'MISSING  ') + f)
print('\nIf any are MISSING, upload them (Option B) before continuing.')

## Cell 3 — Generate the teacher targets (`gen_teacher.py`)

Runs full SDXL on the GPU to produce **polished target PNGs** across every category into
`data/caellum_teacher/`. These are the "finished" side of each training pair. All images are
**512×512, white background**. This is the slow step (lots of diffusion) — start it early.

In [ ]:
# Polished targets -> data/caellum_teacher/  (uses the CATEGORIES inlined in the script).
# --per-category 10 keeps the first run fast: 35 cats x 10 = 350 imgs (~45-60 min on a T4).
# Raise it for more data (the script default is 24); lower it if you're tight on time.
!python data/gen_teacher.py --per-category 10

import glob
n = len(glob.glob('data/caellum_teacher/**/*.png', recursive=True))
print(f'\nteacher PNGs on disk: {n}')
assert n > 0, 'gen_teacher.py produced no PNGs — check the cell output above.'

## Cell 4 — Roughen into paired dataset (`roughen.py`)

Turns each finished PNG into a crude "kid's sketch" (edge → simplify → jitter → drop strokes) and
writes the **paired dataset** to `data/caellum_pairs/`:
`rough/<id>.png`, `finished/<id>.png`, and `metadata.jsonl` (one
`{id, rough, finished, label, edit_prompt}` row per pair). This is exactly what the training
script reads. Still 512×512, white background — background removal happens at *serve* time, not
here.

In [ ]:
# Finished PNGs -> rough/finished pairs + metadata.jsonl in data/caellum_pairs/
!python data/roughen.py

import json, os
meta = 'data/caellum_pairs/metadata.jsonl'
assert os.path.exists(meta), 'roughen.py did not write metadata.jsonl — check output above.'
with open(meta) as fh:
    rows = [json.loads(line) for line in fh if line.strip()]
print(f'dataset rows: {len(rows)}')
print('example row:', rows[0] if rows else '(empty!)')

## Cell 5 — Train (InstructPix2Pix, primary)

`accelerate launch` the InstructPix2Pix fine-tune on the pairs
(`original_image=ROUGH`, `edited_image=FINISHED`, `edit_prompt="turn this into a clean game
{label}"`). Output weights go to `caellum_ip2p_out/`.

**Fallback:** if this OOMs or you're short on GPU time, run the lighter img2img LoRA instead —
swap the command for:
```
!accelerate launch train/train_caellum_lora.py \
    --data_dir data/caellum_pairs --output_dir caellum_lora_out --resolution 512
```
(Upload `train/train_caellum_lora.py` in Cell 2 if you go this route.)

In [ ]:
OUTPUT_DIR = 'caellum_ip2p_out'

# Primary: InstructPix2Pix fine-tune on the rough->finished pairs.
!accelerate launch train/train_caellum_ip2p.py \
    --data_dir data/caellum_pairs \
    --output_dir {OUTPUT_DIR} \
    --resolution 512

import os
assert os.path.isdir(OUTPUT_DIR) and os.listdir(OUTPUT_DIR), \
    f'{OUTPUT_DIR} is empty — training did not produce weights. See output above (try the LoRA fallback).'
print('trained weights in', OUTPUT_DIR, ':', os.listdir(OUTPUT_DIR))

## Cell 6 — Zip + download the trained weights

Bundle the output dir and download it. Take this `.zip` to the **Neuron box** and continue at
step 6 of [`RUNBOOK-image.md`](../RUNBOOK-image.md):
```
python compile.py --base <best> --lora <unzipped_dir>
```
which fuses the style into the `caellum_neuron/` artifact at compile time.

In [ ]:
import shutil, os

OUTPUT_DIR = 'caellum_ip2p_out'      # change to 'caellum_lora_out' if you used the LoRA fallback
ZIP_BASE = 'caellum_trained'

zip_path = shutil.make_archive(ZIP_BASE, 'zip', OUTPUT_DIR)
print('zipped ->', zip_path, f'({os.path.getsize(zip_path) / 1e6:.1f} MB)')

try:
    from google.colab import files
    files.download(zip_path)
except Exception as e:
    print('Not on Colab (or download blocked):', e)
    print('Download manually from the file pane:', zip_path)